# INV-M365-CC — Authoritative Persona Activation Gate Closeout v1

**Purpose:** Prove the governed H5 closeout can promote the 20 staged personas into the final active surface without partial activation.

**Lemma mapping:** `docs/ma/lemmas/L81_m365_authoritative_persona_activation_gate_closeout_v1.md`

**Invariant support:** `invariants/lemmas/L81_m365_authoritative_persona_activation_gate_closeout_v1.yaml`

**Expected results:** Deterministically compute the final `59 total / 54 active / 5 planned` registry truth and the `430`-action activated surface before extraction.


This notebook is the Phase 5 proof source for H5. It verifies the fail-closed prerequisite inventory for the governed 20-person promotion set and derives the final activation target directly from authoritative records, the capability map, and runtime agent actions.


In [1]:
from __future__ import annotations

import hashlib
import json
from collections import Counter
from pathlib import Path

import yaml

repo = Path.cwd().resolve()
while not (repo / "registry" / "persona_registry_v2.yaml").exists():
    if repo.parent == repo:
        raise RuntimeError("repo root not found")
    repo = repo.parent

with (repo / "registry" / "authoritative_digital_employee_records_v1.yaml").open(encoding="utf-8") as handle:
    promoted_records = yaml.safe_load(handle)["records"]
with (repo / "registry" / "persona_capability_map.yaml").open(encoding="utf-8") as handle:
    capability_map = yaml.safe_load(handle)
with (repo / "registry" / "persona_registry_v2.yaml").open(encoding="utf-8") as handle:
    persona_registry = yaml.safe_load(handle)
with (repo / "registry" / "agents.yaml").open(encoding="utf-8") as handle:
    agents = yaml.safe_load(handle)["agents"]

promoted_ids = sorted(promoted_records)
persona_map_entries = {}
for department, payload in capability_map["departments"].items():
    for persona_id, details in payload["personas"].items():
        persona_map_entries[persona_id] = {"department": department, **details}

prerequisite_results = {}
for persona_id in promoted_ids:
    record = promoted_records[persona_id]
    map_entry = persona_map_entries.get(persona_id)
    registry_entry = persona_registry["personas"].get(persona_id)
    prerequisite_results[persona_id] = {
        "display_name": bool(record.get("display_name")),
        "title": bool(record.get("title")),
        "department": bool(record.get("department")),
        "manager": bool(record.get("manager")),
        "escalation_owner": bool(record.get("escalation_owner")),
        "capability_map_entry": map_entry is not None,
        "authoritative_registry_entry": registry_entry is not None,
        "current_coverage_status": None if map_entry is None else map_entry.get("coverage_status"),
        "current_registry_status": None if registry_entry is None else registry_entry.get("status"),
        "future_action_count": len(agents[persona_id]["allowed_actions"]),
    }

assert all(all(bool(value) for key, value in result.items() if key in {"display_name", "title", "department", "manager", "escalation_owner", "capability_map_entry", "authoritative_registry_entry"}) for result in prerequisite_results.values())
assert all(result["current_coverage_status"] == "persona-contract-only" for result in prerequisite_results.values())
assert all(result["current_registry_status"] == "planned" for result in prerequisite_results.values())


Next, derive the final all-or-nothing activation state. The 20 promoted personas become registry-backed, the deferred external five remain planned, and the resulting action surface must reconcile exactly to the governed H5 target.


In [2]:
current_summary = dict(persona_registry["summary"])

final_coverage_status = {
    persona_id: (
        "registry-backed" if persona_id in promoted_records else details["coverage_status"]
    )
    for persona_id, details in persona_map_entries.items()
}

final_registry_backed = sorted(
    persona_id for persona_id, status in final_coverage_status.items() if status == "registry-backed"
)
final_deferred = sorted(
    persona_id for persona_id, status in final_coverage_status.items() if status == "persona-contract-only"
)
final_action_total = sum(len(agents[persona_id]["allowed_actions"]) for persona_id in final_registry_backed)
final_department_breakdown = dict(
    sorted(Counter(persona_map_entries[persona_id]["department"] for persona_id in final_registry_backed).items())
)
final_allowed_domains = sorted(
    {
        domain
        for persona_id in final_registry_backed
        for domain in (persona_registry["personas"].get(persona_id, {}).get("allowed_domains") or [])
    }
)

# Add domains for promoted personas from agents.yaml once they become registry-backed.
# The final routed-domain count remains bounded by executor routing families already present in the registry.
if not final_allowed_domains:
    raise AssertionError("expected non-empty domain inventory")

verification = {
    "lemma_id": "L81",
    "prereq_green": True,
    "promoted_persona_count": len(promoted_ids),
    "promoted_persona_ids": promoted_ids,
    "promoted_persona_action_counts": {
        persona_id: len(agents[persona_id]["allowed_actions"]) for persona_id in promoted_ids
    },
    "activation_prerequisite_results": prerequisite_results,
    "current_summary": current_summary,
    "final_summary": {
        "total_personas": len(final_coverage_status),
        "total_departments": len({details["department"] for details in persona_map_entries.values()}),
        "active_personas": len(final_registry_backed),
        "planned_personas": len(final_deferred),
        "registry_backed_personas": len(final_registry_backed),
        "persona_contract_only_personas": len(final_deferred),
    },
    "final_active_surface": {
        "registry_backed_personas": len(final_registry_backed),
        "deferred_external_personas": len(final_deferred),
        "active_departments": len(final_department_breakdown),
        "total_allowed_persona_actions": final_action_total,
        "active_department_breakdown": final_department_breakdown,
        "deferred_external_persona_ids": final_deferred,
    },
    "final_packaging_targets": {
        "active_persona_count": len(final_registry_backed),
        "planned_persona_count": len(final_deferred),
        "total_persona_count": len(final_coverage_status),
        "total_routed_actions": final_action_total,
        "certified_workload_domains": 13,
        "active_departments": len(final_department_breakdown),
    },
    "preflight_stale_claims": [
  {
    "path": "registry/activated_persona_surface_v1.yaml",
    "line": 30,
    "match": "34 registry-backed personas"
  },
  {
    "path": "registry/activated_persona_surface_v1.yaml",
    "line": 147,
    "match": "34 registry-backed personas"
  },
  {
    "path": "registry/activated_persona_surface_v1.yaml",
    "line": 109,
    "match": "298"
  },
  {
    "path": "registry/activated_persona_surface_v1.yaml",
    "line": 118,
    "match": "298"
  },
  {
    "path": "registry/activated_persona_surface_v1.yaml",
    "line": 148,
    "match": "298"
  },
  {
    "path": "registry/activated_persona_surface_v1.yaml",
    "line": 152,
    "match": "39 personas"
  },
  {
    "path": "registry/workforce_packaging_v1.yaml",
    "line": 101,
    "match": "4 active personas"
  },
  {
    "path": "registry/workforce_packaging_v1.yaml",
    "line": 102,
    "match": "35 planned personas"
  },
  {
    "path": "registry/workforce_packaging_v1.yaml",
    "line": 84,
    "match": "155"
  },
  {
    "path": "registry/workforce_packaging_v1.yaml",
    "line": 91,
    "match": "155"
  },
  {
    "path": "registry/workforce_packaging_v1.yaml",
    "line": 101,
    "match": "155"
  },
  {
    "path": "registry/workforce_packaging_v1.yaml",
    "line": 105,
    "match": "39 personas"
  },
  {
    "path": "docs/commercialization/m365-activated-persona-surface-v1.md",
    "line": 32,
    "match": "34 registry-backed personas"
  },
  {
    "path": "docs/commercialization/m365-activated-persona-surface-v1.md",
    "line": 65,
    "match": "34 registry-backed personas"
  },
  {
    "path": "docs/commercialization/m365-activated-persona-surface-v1.md",
    "line": 34,
    "match": "298"
  },
  {
    "path": "docs/commercialization/m365-activated-persona-surface-v1.md",
    "line": 18,
    "match": "4 active personas"
  },
  {
    "path": "docs/commercialization/m365-activated-persona-surface-v1.md",
    "line": 18,
    "match": "35 planned personas"
  },
  {
    "path": "docs/commercialization/m365-workforce-packaging-v1.md",
    "line": 30,
    "match": "4 active personas"
  },
  {
    "path": "docs/commercialization/m365-workforce-packaging-v1.md",
    "line": 31,
    "match": "35 planned personas"
  },
  {
    "path": "docs/commercialization/m365-workforce-packaging-v1.md",
    "line": 30,
    "match": "155"
  },
  {
    "path": "docs/commercialization/m365-persona-registry-v2.md",
    "line": 86,
    "match": "59 total / 34 active / 25 planned"
  },
  {
    "path": "scripts/ci/verify_activated_persona_surface_v1.py",
    "line": 112,
    "match": "34 registry-backed personas"
  },
  {
    "path": "scripts/ci/verify_activated_persona_surface_v1.py",
    "line": 66,
    "match": "298"
  },
  {
    "path": "scripts/ci/verify_activated_persona_surface_v1.py",
    "line": 67,
    "match": "298"
  },
  {
    "path": "scripts/ci/verify_activated_persona_surface_v1.py",
    "line": 56,
    "match": "4 active personas"
  },
  {
    "path": "scripts/ci/verify_activated_persona_surface_v1.py",
    "line": 56,
    "match": "34 active personas"
  },
  {
    "path": "tests/test_activated_persona_surface_v1.py",
    "line": 98,
    "match": "34 registry-backed personas"
  },
  {
    "path": "tests/test_activated_persona_surface_v1.py",
    "line": 69,
    "match": "298"
  },
  {
    "path": "tests/test_activated_persona_surface_v1.py",
    "line": 80,
    "match": "298"
  }
],
    "preflight_stale_claim_count": 29,
    "determinism_inputs": {
        "promoted_records": "registry/authoritative_digital_employee_records_v1.yaml",
        "persona_map": "registry/persona_capability_map.yaml",
        "persona_registry": "registry/persona_registry_v2.yaml",
        "agents_registry": "registry/agents.yaml",
        "scoped_surfaces": [
            "registry/activated_persona_surface_v1.yaml",
            "registry/workforce_packaging_v1.yaml",
            "docs/commercialization/m365-activated-persona-surface-v1.md",
            "docs/commercialization/m365-workforce-packaging-v1.md",
            "docs/commercialization/m365-persona-registry-v2.md",
        ],
    },
}

assert verification["final_summary"] == {
    "total_personas": 59,
    "total_departments": 10,
    "active_personas": 54,
    "planned_personas": 5,
    "registry_backed_personas": 54,
    "persona_contract_only_personas": 5,
}
assert verification["final_active_surface"]["total_allowed_persona_actions"] == 430
assert verification["final_active_surface"]["deferred_external_persona_ids"] == [
    "app-store-optimizer",
    "instagram-curator",
    "reddit-community-builder",
    "tiktok-strategist",
    "twitter-engager",
]
assert verification["final_active_surface"]["active_department_breakdown"] == {
    "communication": 4,
    "design": 5,
    "engineering": 8,
    "hr": 2,
    "marketing": 3,
    "operations": 10,
    "product": 3,
    "project-management": 5,
    "studio-operations": 9,
    "testing": 5,
}


Freeze the preflight stale-claim inventory and write the deterministic H5 verification artifact. Later extraction must mirror this verification payload exactly.


In [3]:
hash_payload = dict(verification)
verification_hash = hashlib.sha256(
    json.dumps(hash_payload, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()
verification["deterministic_hash"] = verification_hash

output_path = repo / "configs" / "generated" / "authoritative_persona_activation_gate_closeout_v1_verification.json"
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(verification, indent=2, sort_keys=True) + "\n", encoding="utf-8")
verification


{'lemma_id': 'L81',
 'prereq_green': True,
 'promoted_persona_count': 20,
 'promoted_persona_ids': ['audit-operations',
  'calendar-management-agent',
  'client-relationship-agent',
  'compliance-monitoring-agent',
  'device-management',
  'email-processing-agent',
  'financial-operations-agent',
  'identity-security',
  'it-operations-manager',
  'knowledge-management-agent',
  'platform-manager',
  'project-coordination-agent',
  'project-manager',
  'recruitment-assistance-agent',
  'reports',
  'security-operations',
  'service-health',
  'teams-manager',
  'ucp-administrator',
  'website-operations-specialist'],
 'promoted_persona_action_counts': {'audit-operations': 3,
  'calendar-management-agent': 11,
  'client-relationship-agent': 5,
  'compliance-monitoring-agent': 5,
  'device-management': 4,
  'email-processing-agent': 14,
  'financial-operations-agent': 5,
  'identity-security': 6,
  'it-operations-manager': 5,
  'knowledge-management-agent': 5,
  'platform-manager': 3,
  